# Letterboxd → TMDB scrape (v2, self-contained)

Single notebook. Run cells top to bottom. No separate worker module to keep in sync.

**Output:** `letterboxd_reviews.csv` with columns
`username, tmdb_id, movie_title, rating, genre, subgenre_1, subgenre_2, subgenre_3, subgenre_4, subgenre_5`

Three phases, all checkpointed to JSONL files so you can rerun the cell to resume:

1. Scrape every user's `/films/` page → `phase1.jsonl`
2. Resolve unique slugs to TMDB ids → `phase2.jsonl`
3. Fetch genres from TMDB API → `phase3.jsonl`
4. Merge → `letterboxd_reviews.csv`


In [ ]:
# ===== 1. Install dependencies =====
# curl_cffi impersonates a real Chrome TLS handshake so Cloudflare lets us through.
# Re-running this cell is a no-op once installed.
import sys, subprocess
for pkg in ["curl_cffi", "beautifulsoup4", "pandas", "tqdm"]:
    try:
        __import__("bs4" if pkg == "beautifulsoup4" else pkg.replace("_", "-").replace("-", "_"))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("deps OK")


In [ ]:
# ===== 2. Configuration =====
TMDB_API_KEY  = "PASTE_YOUR_TMDB_V3_API_KEY_HERE"   # https://www.themoviedb.org/settings/api
USERNAMES_CSV = "usernames.csv"

WORKERS_LBX   = 16     # threads hitting Letterboxd. Lower if you see lots of 403/429.
WORKERS_TMDB  = 30     # threads hitting TMDB API.

MAX_USERS     = None   # set to e.g. 25 for a quick test, None = all
WIPE_AND_RESTART = False   # set True ONCE if you want to nuke phase1.jsonl and start over

from pathlib import Path
import os
PHASE1 = Path("phase1.jsonl")
PHASE2 = Path("phase2.jsonl")
PHASE3 = Path("phase3.jsonl")
FINAL  = Path("letterboxd_reviews.csv")

if WIPE_AND_RESTART:
    for p in (PHASE1, PHASE2, PHASE3, FINAL):
        if p.exists():
            p.unlink()
            print("removed", p)
print("config OK; workers =", WORKERS_LBX, "/", WORKERS_TMDB)


In [ ]:
# ===== 3. Imports and HTTP helpers =====
import re, json, time, threading
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from bs4 import BeautifulSoup
from curl_cffi import requests
from tqdm.auto import tqdm

# Each worker thread gets its own curl_cffi Session (TLS reuse + impersonation).
_tls = threading.local()
def session():
    s = getattr(_tls, "s", None)
    if s is None:
        s = requests.Session(impersonate="chrome124")
        s.headers.update({
            "Accept-Language": "en-US,en;q=0.9",
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        })
        _tls.s = s
    return s

def get_with_retry(url, *, max_retries=5, timeout=25):
    backoff = 2.0
    for _ in range(max_retries):
        try:
            r = session().get(url, timeout=timeout)
        except Exception:
            time.sleep(backoff); backoff *= 2; continue
        if r.status_code == 200:
            # detect Cloudflare interstitial ("Just a moment...")
            if len(r.text) < 8000 and "Just a moment" in r.text:
                time.sleep(backoff); backoff *= 2; continue
            return r
        if r.status_code == 404:
            return r
        if r.status_code in (403, 429, 500, 502, 503, 504):
            time.sleep(backoff); backoff *= 2; continue
        return r
    return None


In [ ]:
# ===== 4. Scraping primitives =====
RATING_RE    = re.compile(r"rated-(\d+)")
TMDB_LINK_RE = re.compile(r"themoviedb\.org/movie/(\d+)")

def scrape_user_films(username, max_pages=1000):
    """Yields {username, slug, title, rating} for every film the user logged."""
    out, seen = [], set()
    page = 1
    while page <= max_pages:
        r = get_with_retry(f"https://letterboxd.com/{username}/films/page/{page}/")
        if r is None or r.status_code != 200:
            break
        soup = BeautifulSoup(r.text, "html.parser")
        items = (soup.select("li.poster-container")
                 or soup.select("li.griditem")
                 or soup.select("li[data-owner-rating], li[data-film-id]"))
        if not items:
            break
        new = 0
        for item in items:
            poster = item.select_one(
                "div[data-film-slug], div[data-target-link^='/film/'], "
                "div.film-poster, div.really-lazy-load"
            )
            if poster is None and item.get("data-film-slug"):
                poster = item
            if poster is None:
                continue
            slug = poster.get("data-film-slug")
            if not slug:
                target = poster.get("data-target-link") or ""
                if target.startswith("/film/"):
                    slug = target.strip("/").split("/")[-1]
            if not slug or slug in seen:
                continue
            seen.add(slug)
            img = poster.select_one("img")
            title = (img.get("alt") if img else None) or slug.replace("-", " ").title()
            rating = None
            owner = item.get("data-owner-rating")
            if owner and owner not in ("0", ""):
                try: rating = int(owner) / 2.0
                except (TypeError, ValueError): pass
            if rating is None:
                rs = item.select_one("span.rating")
                if rs:
                    m = RATING_RE.search(" ".join(rs.get("class", [])))
                    if m: rating = int(m.group(1)) / 2.0
            out.append({"username": username, "slug": slug, "title": title, "rating": rating})
            new += 1
        if new == 0 or not soup.select_one("div.paginate-nextprev a.next"):
            break
        page += 1
    return out


def resolve_tmdb_id(slug):
    r = get_with_retry(f"https://letterboxd.com/film/{slug}/")
    if r is None or r.status_code != 200:
        return {"slug": slug, "tmdb_id": None, "title": None}
    m = TMDB_LINK_RE.search(r.text)
    tmdb_id = int(m.group(1)) if m else None
    soup = BeautifulSoup(r.text, "html.parser")
    title_tag = soup.select_one("h1.headline-1, h1.filmtitle, h1")
    title = title_tag.get_text(strip=True) if title_tag else None
    return {"slug": slug, "tmdb_id": tmdb_id, "title": title}


def fetch_tmdb(tmdb_id, api_key):
    url = f"https://api.themoviedb.org/3/movie/{tmdb_id}?api_key={api_key}&language=en-US"
    backoff = 1.5
    for _ in range(5):
        try:
            r = session().get(url, timeout=20)
        except Exception:
            time.sleep(backoff); backoff *= 2; continue
        if r.status_code == 200:
            d = r.json()
            return {"tmdb_id": tmdb_id,
                    "title": d.get("title") or d.get("original_title"),
                    "genres": [g["name"] for g in d.get("genres", [])]}
        if r.status_code == 429:
            time.sleep(float(r.headers.get("Retry-After", backoff))); backoff *= 2; continue
        if r.status_code in (500, 502, 503, 504):
            time.sleep(backoff); backoff *= 2; continue
        return {"tmdb_id": tmdb_id, "title": None, "genres": []}
    return {"tmdb_id": tmdb_id, "title": None, "genres": []}


## Connectivity check — must show >0 films before continuing

If this fails with 0 films:
- Make sure the `pip install` cell finished successfully.
- If status is 403 with `"Just a moment..."` in the body, your IP is being challenged by Cloudflare. Wait a few minutes and try again, or open `letterboxd.com` in your browser to clear the challenge.

In [ ]:
_films = scrape_user_films("schaffrillas", max_pages=1)
print(f"got {len(_films)} films from schaffrillas page 1")
print("sample:", _films[:3])
assert len(_films) > 0, "Letterboxd scrape returned 0 films - fix this before continuing"


In [ ]:
# Optional: only run this cell if you want to verify your TMDB API key
_g = fetch_tmdb(496243, TMDB_API_KEY)   # Parasite (2019)
print("TMDB sample:", _g)
assert _g["genres"], "TMDB returned no genres - check TMDB_API_KEY"


## Load usernames

In [ ]:
usernames = (
    pd.read_csv(USERNAMES_CSV)["username"]
    .dropna().astype(str).str.strip().str.lower()
    .unique().tolist()
)
if MAX_USERS:
    usernames = usernames[:MAX_USERS]
print(f"{len(usernames):,} usernames queued")


## Phase 1 — scrape every user's `/films/` page

Re-running this cell skips users that already have at least one film recorded in `phase1.jsonl`.
Empty entries are NOT counted as done, so a stale checkpoint won't trap you.

In [ ]:
def phase1_done() -> set:
    done = set()
    if not PHASE1.exists():
        return done
    with PHASE1.open() as fh:
        for line in fh:
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue
            # Only count as done if we actually got films (so empty stale entries get retried)
            if rec.get("films"):
                done.add(rec["username"])
    return done


def run_phase1(usernames, workers):
    done = phase1_done()
    todo = [u for u in usernames if u not in done]
    print(f"Phase 1: {len(done):,} already have films, {len(todo):,} to scrape")
    if not todo:
        return
    lock = threading.Lock()
    fh = PHASE1.open("a", buffering=1)   # line-buffered

    def worker(u):
        try:
            films = scrape_user_films(u)
        except Exception as e:
            films = []
        rec = {"username": u, "films": films}
        with lock:
            fh.write(json.dumps(rec) + "\n")
        return len(films)

    try:
        with ThreadPoolExecutor(max_workers=workers) as ex:
            futures = [ex.submit(worker, u) for u in todo]
            counts = []
            for fut in tqdm(as_completed(futures), total=len(todo), desc="Phase 1"):
                counts.append(fut.result())
        print(f"avg films/user: {sum(counts)/max(len(counts),1):.1f}")
    finally:
        fh.close()

run_phase1(usernames, WORKERS_LBX)


## Aggregate Phase 1 → unique slugs

In [ ]:
rows = []
with PHASE1.open() as fh:
    for line in fh:
        try:
            rec = json.loads(line)
        except json.JSONDecodeError:
            continue
        rows.extend(rec.get("films", []))

films_df = pd.DataFrame(rows)
print(f"{len(films_df):,} (user, film) rows across {films_df['username'].nunique():,} users")
unique_slugs = films_df["slug"].dropna().unique().tolist()
print(f"{len(unique_slugs):,} unique slugs to resolve")
films_df.head()


## Phase 2 — resolve each slug to a TMDB id

In [ ]:
def phase2_done() -> set:
    done = set()
    if not PHASE2.exists():
        return done
    with PHASE2.open() as fh:
        for line in fh:
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue
            done.add(rec["slug"])
    return done


def run_phase2(slugs, workers):
    done = phase2_done()
    todo = [s for s in slugs if s not in done]
    print(f"Phase 2: {len(done):,} resolved, {len(todo):,} to go")
    if not todo:
        return
    lock = threading.Lock()
    fh = PHASE2.open("a", buffering=1)
    def worker(s):
        try:
            return resolve_tmdb_id(s)
        except Exception:
            return {"slug": s, "tmdb_id": None, "title": None}
    try:
        with ThreadPoolExecutor(max_workers=workers) as ex:
            futures = [ex.submit(worker, s) for s in todo]
            for fut in tqdm(as_completed(futures), total=len(todo), desc="Phase 2"):
                rec = fut.result()
                with lock:
                    fh.write(json.dumps(rec) + "\n")
    finally:
        fh.close()

run_phase2(unique_slugs, WORKERS_LBX)


## Aggregate Phase 2 → unique TMDB ids

In [ ]:
slug_rows = []
with PHASE2.open() as fh:
    for line in fh:
        try:
            slug_rows.append(json.loads(line))
        except json.JSONDecodeError:
            continue
slug_df = pd.DataFrame(slug_rows)
slug_df["tmdb_id"] = pd.to_numeric(slug_df["tmdb_id"], errors="coerce").astype("Int64")
print(f"{slug_df['tmdb_id'].notna().sum():,} / {len(slug_df):,} slugs got a TMDB id")
unique_tmdb_ids = slug_df["tmdb_id"].dropna().astype(int).unique().tolist()
print(f"{len(unique_tmdb_ids):,} unique TMDB ids")


## Phase 3 — fetch genres from TMDB

In [ ]:
def phase3_done() -> set:
    done = set()
    if not PHASE3.exists():
        return done
    with PHASE3.open() as fh:
        for line in fh:
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue
            done.add(int(rec["tmdb_id"]))
    return done


def run_phase3(tmdb_ids, api_key, workers):
    if not api_key or api_key.startswith("PASTE_"):
        raise RuntimeError("Set TMDB_API_KEY in the config cell first.")
    done = phase3_done()
    todo = [t for t in tmdb_ids if t not in done]
    print(f"Phase 3: {len(done):,} fetched, {len(todo):,} to go")
    if not todo:
        return
    lock = threading.Lock()
    fh = PHASE3.open("a", buffering=1)
    def worker(t):
        try:
            return fetch_tmdb(t, api_key)
        except Exception:
            return {"tmdb_id": t, "title": None, "genres": []}
    try:
        with ThreadPoolExecutor(max_workers=workers) as ex:
            futures = [ex.submit(worker, t) for t in todo]
            for fut in tqdm(as_completed(futures), total=len(todo), desc="Phase 3"):
                rec = fut.result()
                with lock:
                    fh.write(json.dumps(rec) + "\n")
    finally:
        fh.close()

run_phase3(unique_tmdb_ids, TMDB_API_KEY, WORKERS_TMDB)


## Final merge → CSV

In [ ]:
genre_rows = []
with PHASE3.open() as fh:
    for line in fh:
        try:
            genre_rows.append(json.loads(line))
        except json.JSONDecodeError:
            continue
genre_df = pd.DataFrame(genre_rows)
genre_df["tmdb_id"] = genre_df["tmdb_id"].astype("Int64")

merged = (
    films_df.merge(slug_df[["slug", "tmdb_id"]], on="slug", how="left")
            .merge(genre_df[["tmdb_id", "title", "genres"]], on="tmdb_id", how="left",
                   suffixes=("", "_tmdb"))
)
merged["movie_title"] = merged["title_tmdb"].fillna(merged["title"])

def split_genres(gs):
    gs = gs if isinstance(gs, list) else []
    gs = (gs + [None] * 6)[:6]
    return pd.Series(gs, index=["genre", "subgenre_1", "subgenre_2", "subgenre_3", "subgenre_4", "subgenre_5"])

genre_cols = merged["genres"].apply(split_genres)
final = pd.concat([merged[["username", "tmdb_id", "movie_title", "rating"]], genre_cols], axis=1)

final.to_csv(FINAL, index=False)
size_mb = FINAL.stat().st_size / 1024 / 1024
print(f"Wrote {FINAL.resolve()}  ({len(final):,} rows, {size_mb:.1f} MB)")
final.head()


In [ ]:
print("Rows                :", f"{len(final):,}")
print("Unique users        :", f"{final['username'].nunique():,}")
print("Unique movies (tmdb):", f"{final['tmdb_id'].nunique():,}")
print("With rating         :", f"{final['rating'].notna().sum():,}")
print("With tmdb_id        :", f"{final['tmdb_id'].notna().sum():,}")
print("With at least 1 genre:", f"{final['genre'].notna().sum():,}")
print()
print("Top genres:")
print(final["genre"].value_counts().head(10))
